## Aerodynamic Resistance Calculation ##

**Based on the book, Handbook for Transversely Finned Tube Heat Exchangers.**

### Pressure drop correlation

The aerodynamic resistance of bundles of finned tubes in crossflow is determined from the equation:

$\Delta P = C_{op} \cdot \zeta_0 \cdot z_2 \cdot \frac{\rho_g u_g^2}{2} $

Here, 

$C_{op}$: Correction factor. Taken to be $C_{op} = 1.1$

$\zeta_o$: Resistance coefficient


### Calculation of equivalent Reynold's number

The method is valid for $Re_{eq} = 5 \cdot 10 ^3$ to $6 \cdot 10^4$

This is calculated as $\frac{u_g d_{eq}}{v_g}$

$u_g$ is the design gas velocity: $u_g = \frac{G_g}{F \rho_g}=\frac{G_g v_g}{F}$

Where, 

$G_g$: Mass flow rate of external heat-transfer medium 

$F$: Minimum flow area 

The minimum free flow area is calculated depending on the $\varphi_{cl}$ parameter: $\varphi_{cl} = \frac{S_1-d_{cl}}{S_1' - d_{cl}}$

$d_{cl}$ is the conventional diameter of the finned tube: $d_{cl} = d + \frac{2 l_r \delta_r}{S_r}$

Where, 

$l_r$: fin height

$\delta_r$: average fin thickness

$S_r$: fin spacing

$d$: outside diameter of fin carrying tube.

The minimum free flow area for $\varphi_{cl} \leq 2$: $F=a \cdot b - z_1 L_{c\cdot cr \cdot s} \cdot d_{cl}$

The minimum free flow area for $\varphi_{cl} > 2$: $F=(a \cdot b - z_1 L_{c\cdot cr \cdot s} \cdot d_{cl}) \frac{2}{\varphi_{cl}}$

Where,

$L_{c\cdot cr \cdot s}$: Length of pipe at flow cross section

$z_1$: Number of tubes in transverse row of bundle

$a$ and $b$ are the dimensions of the gas conduit / cross section. So height and breadth. 

### Calculation of Resistance Coefficient ###

The resistance coefficient is calculated from: $\zeta_0 = C_z' \cdot C_r \cdot (\frac{u_g d_{eq}}{v_g})^{-n}$

For staggered arrangement, the exponent $n$ and coefficient $C_r$ are calculated as:

$n=0.17 (\frac{A_{total}}{F})^{0.25} (\frac{S_1}{S_2})^{0.57} \exp{(-0.36\frac{S_1}{S_2})}$

$C_r=2.8 (\frac{A_{total}}{F})^{0.53} (\frac{S_1}{S_2})^{1.30} \exp{(-0.90\frac{S_1}{S_2})}$

For inline arrangement, the exponent $n$ and coefficient $C_r$ are calculated as:

For $\frac{S_1}{S_2}\leq 2,1$

$n=(\frac{A_{total}}{F})^{0.08}(0.184-0.088 \frac{S_1}{S_2})$

$C_r = 2.5(\frac{A_total}{F})^{0.25} \exp{(-1.70\frac{S_1}{S_2})}$


For $\frac{S_1}{S_2} > 2,1$

$n=0$

$C_r = (\frac{A_total}{F})^{0.10} (0.132 - 0.016 \frac{S_1}{S_2})$


Where, 

$S_1$: Transverse spacing of tubes

$S_2$: Longitudinal spacing of tubes

The quantity $\frac{A_{total}}{F}=\frac{\pi\cdot (d\cdot s_r + 2 l_r \cdot \delta_r + 2 \cdot l_r \cdot (l+d))}{S_1 \cdot s_r - (d s_r + 2 l_r \delta_r)}$ is called the reduced length of the developed surface.  

### Equivalent diameter

The equivalent diameter of the most contracted cross-section is calculated depending on $\varphi_{cl}$.

For staggered and inline bundles with $\varphi_{cl} \leq 2$

$d_{eq}=\frac{2(s_r\cdot(S_1-d)-2l_r \delta_r)}{2 l_r + s_r}$

For staggered bundles with $\varphi_{cl} > 2$

$d_{eq}'=\frac{2 d_{eq}}{\varphi_{cl}}$

For square fins, the following must be assumed:

$l_r = l_A = (1.13 c_{sq}-d)0.5$

### Correction factor for small row numbers

For staggered arrangement and $z_2 < 6$ the correction factor is:

$C_z' = \exp{(0.1(\frac{6}{z_2}-1))}$

For inline arrangement and $z_2 < 6$

$C_z' = 1 + \frac{0.65}{(z_2)^3}$

For any arrangment of tubes and $z_2 \geq 6$

$C_z' \approx 1.0$

## Python model

Packs:

In [2]:
import numpy as np

import pyfluids as pf
from pyfluids import FluidsList, Input,Phases

Test parameters:

In [50]:
#SGH geometry:
# height of SGH
a      = 2.858
# Width of SGH
b      = 3.773

#Fin geometry:

#Fin spacing/pitch:
s_r = 9/1000
#avg fin thickness:
delta_r = 2/1000
#Fin height:
l_r = (27/1000)-((31.8/2)/1000)
#fin width
w = (27/1000)

#Tube geometry:
# Outer tube diameter
d = 31.8/1000
#outside diameter of finning
D = d+2*l_r
#Transverse spacing of tubes:
S_1  = 2*w
#Longitudinal spacing of tubes:
S_2  = 2*w

#number of possible rows:
N = np.floor((-d+S_1+b)/S_1)

#Transverse number of rows:
z_1    = N
#Longitudinal number of rows:
z_2    = 30

#Length of pipe at flow cross section
L_ccrs = b

# Inlet mass flow rate
G_g  = 62.917

# Normal conditions:
T_N    = 15
P_N    = 96600
rho_N  = 1.2888

# Inlet temperature and pressure
P_in    = -4000 + P_N # Pa
T_in    = 230 # degC

# Initial density:
rho_g   = rho_N *(P_in/P_N)*(T_N/T_in)

# Defining flue gas

air = pf.Fluid(FluidsList.Air)
air.update(Input.temperature(T_in),Input.pressure(P_in))
rho_g = air.density

# Kinematic viscosity
mu      = air.kinematic_viscosity

# Steam inlet pressure and temperature. This should be
T_steam     = 360 + 273.15 # K
P_steam     = 13500000 # Pa

arrangement = 'staggered'

nu_g = air.dynamic_viscosity

S_2m = np.sqrt((1/4)*S_1**2 + S_2**2)

print(rho_g)
print(air.specific_heat)

#Molar masses:
M_CO2 = 44.01
M_SO2 = 64.07
M_N2  = 28.013
M_H2O = 18.015
M_O2  = 31.999
M_Ar  = 39.948

x_CO2 = 0.13090
x_SO2 = 0.00004
x_N2  = 0.66196
x_H2O = 0.15841
x_O2  = 0.04092
x_Ar  = 0.00777

mix_M = (
    x_CO2 * M_CO2 +
    x_SO2 * M_SO2 +
    x_N2  * M_N2  +
    x_H2O * M_H2O +
    x_O2  * M_O2  +
    x_Ar  * M_Ar
)

#Mass fractions:
w_CO2 = (M_CO2 * x_CO2)/mix_M
w_SO2 = (M_SO2 * x_SO2)/mix_M
w_N2  = (M_N2 * x_N2)/mix_M
w_H2O = (M_H2O * x_H2O)/mix_M
w_O2  = (M_O2 * x_O2)/mix_M
w_Ar  = (M_Ar * x_Ar)/mix_M

fluegas = pf.Mixture([FluidsList.CarbonDioxide,
                      FluidsList.Nitrogen,
                      FluidsList.Water,
                      FluidsList.Oxygen,],[w_CO2+w_SO2,w_N2,w_H2O,w_O2+w_Ar]).specify_phase(Phases.Gas).with_state(Input.temperature(T_in),Input.pressure(P_in))

rho_g = fluegas.density
print(rho_g)
print(fluegas.specific_heat)
nu_g      = air.dynamic_viscosity


0.6409513010834955
1030.4308327043304
0.6357813921479917
1133.9469545691268


In [5]:
d_in = 0.028
d = 2.5*10**(-3)
l_r = 0.0135
s_r = 0.003
delta_r = 8 * 10**(-4)
a = 0.56
b = 0.5
z_1 = 9

D = d_in + 2*l_r
print(D)

S_1 = a/(z_1+0.5)
print(S_1)

S_2 = np.sqrt(3)/2 * S_1
print(S_2)

S_2m = np.sqrt((1/4)*S_1**2 + S_2**2)
print(S_2m)

L_ccrs=b
v_g=0.488
rho_g = 1/v_g
G_g=2.7
nu_g = 0.9945 * 10**(-5)

0.055
0.05894736842105264
0.051049918538872176
0.05894736842105264



Reynold calculation:

In [12]:
#Reynold number is first calculated.

#Conventional diameter of finned tube:
d_cl = d + ((2*l_r*delta_r)/(s_r))
print(d_cl)

#phi parameter:
phi_cl = (S_1-d_cl)/(S_2m-d_cl)

print(phi_cl)
print(S_2m)
print(S_1)

#Free flow area:
if phi_cl <= 2:
    F = a*b-z_1*L_ccrs*d_cl
elif phi_cl > 2:
    F = (a*b - z_1*L_ccrs*d_cl)*(2/phi_cl)

#Design gas velocity:
u_g     = (G_g)/(F*rho_g)

#Equivalent diameter of finned tube:
d_eq = (2*(s_r*(S_1-d)-2*l_r*delta_r))/(2*l_r + s_r)

if phi_cl <= 2:
    d_eq = d_eq
elif phi_cl > 2:
    d_eq = (2*d_eq)/phi_cl

#Reynold's number is calculated:
Re_eq = (u_g*d_eq)/nu_g

print(u_g)

print(Re_eq)

Re_eq = (u_g*d_eq)/nu_g

0.0097
1.0
0.05894736842105264
0.05894736842105264
5.574783160566956
5521.234794890318


Delta P calculation:

In [13]:
# Resistance coefficient:

#Correction factor for small row numbers:
if arrangement == 'staggered' and z_2 < 6:
    C_zm = np.exp(0.1*(6/z_2-1))
elif arrangement == 'inline' and z_2 < 6:
    C_zm = 1+(0.65/(z_2**3))
else:
    C_zm = 1

# Atotal/F ratio:
A_total_over_F = (np.pi*(d*s_r+2*l_r*delta_r+2*l_r*(l_r+d)))/(S_1*s_r-(d*s_r+2*l_r*delta_r))

# S_1 over S_2 ratio:
S_1_over_S_2 = S_1/S_2

# coefficient in similarity equation for aerodynamic resistance
if arrangement == 'staggered':
    n = 0.17*(A_total_over_F)**(0.25) * (S_1_over_S_2)**(0.57) * np.exp(-0.36*(S_1_over_S_2))
    C_r = 2.8 * (A_total_over_F)**(0.53) * (S_1_over_S_2)**(1.30) * np.exp(-0.90*(S_1_over_S_2))
elif arrangement == 'inline':
    if S_1_over_S_2 <= 2.1:
        n = (A_total_over_F)**(0.08) * (0.184-0.088*(S_1_over_S_2))
        C_r = 2.5 * (A_total_over_F)**(0.25) * np.exp(-1.70*(S_1_over_S_2))
    elif S_1_over_S_2 > 2.1:
        n = 0
        C_r = (A_total_over_F)**(0.10) * (0.132-0.016*(S_1_over_S_2))

# Resistance coefficient is calculated:
zeta_0 = C_zm * C_r * Re_eq**(-n)

# Pressue loss:

# Constant constant
C_op = 1.1

# Delta P calculation:
delta_P = C_op * zeta_0 * z_2 * (rho_g*u_g**2)/2 



NameError: name 'arrangement' is not defined